# Notebook for Microtrader Layer - 07/25/2024

In [1]:
import pandas as pd

# Set option to display all columns
pd.set_option('display.max_columns', None)

In [2]:
data = pd.read_csv('Final-Data-w-Forecast.csv')
data.head()

,timestamp,symbol,mid_price,mean_vol,mean_liq,transaction_cost,time_idx,day_of_week,hour,forecast_1_hour,forecast_2_hours,forecast_3_hours,forecast_1_day,forecast_1_week,forecast_2_weeks,forecast_1_month,rtype,publisher_id,instrument_id,open,high,low,close,volume,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,price,size,RSI,MACD,MACD_signal,MACD_hist,Stoch_k,Stoch_d,OBV,Upper_BB,Middle_BB,Lower_BB,ATR_1,ATR_2,ATR_5,ATR_10,ATR_20,ADX,+DI,-DI,CCI,DLR,TWAP,VWAP,market_liquidity,expected_price,log_return,volatility
0,2022-07-11 08:09:00,MSFT,2.653700e+11,2.116953,137.2,0.000261,35,0,8,0.000024,0.000022,0.000040,0.009919,0.006332,0.008412,0.011984,33,2,7390,265.34,265.40,265.34,265.40,227,265.30,265.40,485,412,265.40,227,48.348107,45.737206,8.329545,37.407661,35.234990,46.448349,-5218.0,2605.804400,827.4355,-950.933400,2102.66,1950.657923,1549.390537,1347.734028,1253.223171,4.848047,48.246003,51.748211,-58.900888,-2.188563,882.470000,308.942939,897,265.40,-2.188563,6.614035
1,2022-07-11 08:10:00,QQQ,2.928900e+11,2.116857,225.8,0.000416,36,0,8,0.000023,0.000022,0.000040,0.013038,0.008159,0.008393,0.009769,33,2,9072,292.91,292.92,292.86,292.86,869,292.86,292.93,4850,4318,292.86,869,48.432693,-11.294140,4.404808,-15.698948,33.758283,45.829148,-4349.0,2605.802982,827.4425,-950.917982,27.52,989.088962,1245.016430,1215.712625,1191.938012,4.739644,48.332004,51.662220,-57.037776,0.098457,866.534595,308.123459,9168,292.93,0.098457,6.190569
2,2022-07-11 08:10:00,MSFT,2.654700e+11,0.683837,250.0,0.000495,37,0,8,0.000018,0.000023,0.000070,0.009919,0.006332,0.008412,0.011984,33,2,7390,265.50,265.50,265.44,265.44,122,265.24,265.65,398,25,265.44,122,48.347552,-58.035460,-8.083246,-49.952214,0.444280,23.145851,-4471.0,2606.524408,821.4340,-963.656408,27.42,508.254481,1001.497144,1096.883363,1133.712112,4.651275,48.245979,51.748255,-45.397000,-0.098306,850.716316,307.819947,423,265.65,-0.098306,5.925451
3,2022-07-11 08:11:00,QQQ,2.929800e+11,0.871908,351.4,0.000687,38,0,8,0.000024,0.000020,0.000040,0.013038,0.008159,0.008393,0.009769,33,2,9072,292.95,293.01,292.95,293.01,538,292.96,293.01,4800,3330,293.01,538,48.445686,-91.795464,-24.825689,-66.969774,0.881956,11.694839,-3933.0,2606.225901,822.8095,-960.606901,27.57,267.912240,806.711715,989.952026,1078.405006,4.555027,48.345112,51.648714,-40.289046,0.098818,836.416154,307.368446,8130,293.01,0.098818,5.920314
4,2022-07-11 08:11:00,SPY,3.861050e+11,-0.091273,771.4,0.001541,39,0,8,0.000024,0.000023,0.000039,0.000000,0.002727,0.004563,0.004936,33,2,10303,386.07,386.14,386.07,386.14,2101,385.98,386.14,3800,3300,386.14,2101,48.799554,-109.770333,-41.814618,-67.955715,2.359646,1.228627,-1832.0,2605.800814,827.4815,-950.837814,93.13,180.521120,663.995372,900.269824,1029.141256,4.414376,48.704056,51.289814,166.666667,0.275993,825.159250,315.730026,7100,386.14,0.275993,5.497524


### Inputs needed: 
1. Hour, Day, Week, Month
2. RSI, MACD
3. Volume, Moving Volatility over 5 mins
4. TC Forecasts
5. Economic Indicators?
6. OHLCV, Historical TC

### Forecasts needed:
1. TC
2. Short term price forecasts
3. Order Book Dynamics
4. Expected Slippage
5. Fill Rate: probability of order fills for limit orders at different price points


### To-Do:
1. Month
2. Moving Volatility
3. Economic Indicators
4. Historical TC
5. Forecasts: Price, OB Dynamics, Slippage, Fill Rate

In [3]:
data['timestamp'] = pd.to_datetime(data['timestamp'])
data['month'] = data['timestamp'].dt.month

In [4]:
data['5_min_volatility'] = data.groupby('symbol')['volatility'].transform(lambda x: x.rolling(window=5).std())

In [5]:
data['5_min_volume'] = data.groupby('symbol')['volume'].transform(lambda x: x.rolling(window=5).sum())

In [6]:
data['5_min_TC'] = data.groupby('symbol')['transaction_cost'].shift(5)

In [7]:
data = data.iloc[35:,:]

# MicroTrader Model

## Transformer Policy

In [ ]:
# !pip uninstall gymnasium shimmy stable-baselines3 -y
!pip install gym==0.26.0 shimmy>=0.2.1 stable-baselines3 torch

In [ ]:
import pandas as pd

# Set option to display all columns
pd.set_option('display.max_columns', None)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class UNetTransformerEncoder(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNetTransformerEncoder, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        
        # Define U-Net layers
        self.encoder1 = self._block(in_channels, 64)
        self.encoder2 = self._block(64, 128)
        self.encoder3 = self._block(128, 256)
        self.encoder4 = self._block(256, 512)

        self.bottleneck = self._block(512, 1024)

        self.decoder4 = self._block(1024 + 512, 512)
        self.decoder3 = self._block(512 + 256, 256)
        self.decoder2 = self._block(256 + 128, 128)
        self.decoder1 = self._block(128 + 64, out_channels)

        # Define Transformer layers
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=features_dim, nhead=8, dropout=0.1)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=6)

    def _block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(F.max_pool1d(enc1, 2))
        enc3 = self.encoder3(F.max_pool1d(enc2, 2))
        enc4 = self.encoder4(F.max_pool1d(enc3, 2))

        bottleneck = self.bottleneck(F.max_pool1d(enc4, 2))

        dec4 = self.decoder4(torch.cat((F.interpolate(bottleneck, scale_factor=2), enc4), dim=1))
        dec3 = self.decoder3(torch.cat((F.interpolate(dec4, scale_factor=2), enc3), dim=1))
        dec2 = self.decoder2(torch.cat((F.interpolate(dec3, scale_factor=2), enc2), dim=1))
        dec1 = self.decoder1(torch.cat((F.interpolate(dec2, scale_factor=2), enc1), dim=1))

        # Transformer encoding
        x = self.transformer_encoder(dec1.permute(2, 0, 1)).permute(1, 2, 0)  # Permute for transformer encoder and back

        return x

In [ ]:
class CustomUNetTransformerModel(BaseFeaturesExtractor):
    def __init__(self, observation_space: spaces.Box, features_dim: int = 256):
        super(CustomUNetTransformerModel, self).__init__(observation_space, features_dim)
        self.embedding = nn.Linear(observation_space.shape[0], features_dim)  # Adapt the input size if necessary
        self.unet_transformer = UNetTransformerEncoder(1, features_dim)

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        x = self.embedding(observations)
        x = x.unsqueeze(1)
        x = self.unet_transformer(x)
        x = x.mean(dim=2)
        return x

In [ ]:
from stable_baselines3.common.policies import ActorCriticPolicy
from gym import spaces

class CustomTransformerPolicy(ActorCriticPolicy):
    def __init__(self, observation_space, action_space, lr_schedule, *args, **kwargs):
        super(CustomTransformerPolicy, self).__init__(observation_space, action_space, lr_schedule, 
                                                      features_extractor_class=CustomUNetTransformerModel, 
                                                      features_extractor_kwargs={'features_dim': 256},
                                                      *args, **kwargs)

# -----------------------------------------------------------------------------------------------------

## Getting Data Ready - Pipeline for Intermediate Forecasting

#### Example Usage:

```python
obs = new_env.reset()
print('Starting predictions')

action, _states = model_high_loaded.predict(obs)
obs, rewards, done, info = new_env.step(action)

print('Done with predictions')

# Render the final state
# Trades will have the schedule
trades = new_env.render()

# Define the columns to forecast: Add more
columns_to_forecast = ['open', 'high', 'low', 'close', 'volume', 'volatility', 'transaction_cost']

micro_input = forecast_values(data, trades, columns_to_forecast) # data: latest data till current step
```

In [ ]:
# This function is for real-time use
# This function assumes your input trade does not contain OHLCV and forecast values
# Use the new dataset for training purpose (skip this step)

import pandas as pd
import statsmodels.api as sm

def forecast_arima(df: pd.DataFrame, orders: dict, steps: int, columns: list) -> pd.DataFrame:
    forecasts = {}
    for column in columns:
        if column in df.columns:
            series = df[column]
            order = orders[column]

            model = sm.tsa.ARIMA(series, order=order)
            model_fit = model.fit()

            forecast = model_fit.forecast(steps=steps)
            forecasts[column] = forecast

            # Create new DataFrame from forecast to shift
            forecast_df = pd.DataFrame({column: forecast})
            forecast_df[f'forecast_{column}_1hr'] = forecast_df[column].shift(-60)
            forecast_df[f'forecast_{column}_3hr'] = forecast_df[column].shift(-180)

            forecasts[f'forecast_{column}_1hr'] = forecast_df[f'forecast_{column}_1hr'].values
            forecasts[f'forecast_{column}_3hr'] = forecast_df[f'forecast_{column}_3hr'].values

    forecast_df = pd.DataFrame(forecasts)
    return forecast_df

def forecast_values(data, trades, columns_to_forecast):
    forecasted_data = pd.DataFrame()
    step_forecast = pd.DataFrame()
    
    # Define ARIMA orders for different columns: Add more
    orders = {
        'open': (60, 1, 0),
        'high': (60, 1, 0),
        'low': (60, 1, 0),
        'close': (60, 1, 0),
        'volume': (60, 1, 0),
        'volatility': (60, 1, 0),
        'transaction_cost': (60, 1, 0)
    }

    max_step = trades['step'].max()
    complete_forecast = forecast_arima(data, orders, max_step+190, columns_to_forecast)
    
    for step in trades['step']:
#         step_forecast = forecast_arima(data, orders, step+370, columns_to_forecast)
#         step_forecast['step'] = step
        step_forecast = complete_forecast.iloc[step - 1]
        forecasted_data = forecasted_data.append(step_forecast, ignore_index=True)
    
    return forecasted_data 


class TechnicalIndicators:
    def __init__(self, data):
        self.data = data

    def add_momentum_indicators(self):
        self.data['RSI'] = ta.RSI(self.data['close'], timeperiod=14)
        self.data['MACD'], self.data['MACD_signal'], self.data['MACD_hist'] = ta.MACD(self.data['close'], fastperiod=12, slowperiod=26, signalperiod=9)
        self.data['Stoch_k'], self.data['Stoch_d'] = ta.STOCH(self.data['high'], self.data['low'], self.data['close'],
                                                              fastk_period=14, slowk_period=3, slowd_period=3)

    def add_volume_indicators(self):
        self.data['OBV'] = ta.OBV(self.data['close'], self.data['volume'])

    def add_volatility_indicators(self):
        self.data['Upper_BB'], self.data['Middle_BB'], self.data['Lower_BB'] = ta.BBANDS(self.data['close'], timeperiod=20)
        self.data['ATR_1'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=1)
        self.data['ATR_2'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=2)
        self.data['ATR_5'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=5)
        self.data['ATR_10'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=10)
        self.data['ATR_20'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=20)
        
    # Adding volatility to calculate dynamic volatility
    def add_volatility(self, window=15):
        self.data['log_return'] = np.log(self.data['close'] / self.data['close'].shift(1))
        self.data['volatility'] = self.data['log_return'].rolling(window=window).std()*np.sqrt(window)

    def add_trend_indicators(self):
        self.data['ADX'] = ta.ADX(self.data['high'], self.data['low'], self.data['close'], timeperiod=14)
        self.data['+DI'] = ta.PLUS_DI(self.data['high'], self.data['low'], self.data['close'], timeperiod=14)
        self.data['-DI'] = ta.MINUS_DI(self.data['high'], self.data['low'], self.data['close'], timeperiod=14)
        self.data['CCI'] = ta.CCI(self.data['high'], self.data['low'], self.data['close'], timeperiod=5)

    def add_other_indicators(self):
        self.data['DLR'] = np.log(self.data['close'] / self.data['close'].shift(1))
        self.data['TWAP'] = self.data['close'].expanding().mean()
        self.data['VWAP'] = (self.data['volume'] * (self.data['high'] + self.data['low']) / 2).cumsum() / self.data['volume'].cumsum()
        self.data['market_liquidity'] = self.data['bid_sz_00'] + self.data['ask_sz_00']
        self.data['expected_price'] = self.data['ask_px_00']

    def add_all_indicators(self):
        self.add_momentum_indicators()
        self.add_volume_indicators()
        self.add_volatility_indicators()
        self.add_trend_indicators()
        self.add_other_indicators()
        self.add_volatility()
        return self.data

def add_technical_indicators(data, forecasted_data):
    # data is the current extracted OHLCV data
    data = data[['open', 'high', 'low', 'close', 'volume', 'volatility', 'transaction_cost']].iloc[-40:]
    complete_data = pd.concat([data,forecasted_data], axis=0) # Complete this
    # Join this with forecasted_data
    indicators = TechnicalIndicators(complete_data)
    data_w_indicators = indicators.add_all_indicators()
    return data_w_indicators.iloc[-40:]
    
def add_volatility(data_w_indicators):
    columns_to_forecast = ['volatility']
    final_data = forecast_values(data, trades, columns_to_forecast)
    return final_data

In [ ]:
def real_time_data_preprocess(data, trades):
    # This function is for preprocessing, adding forecasts and adding indicators to macro output
    # Input: Latest data, trades dictonary returned by macrotrader
    # Output: Dataframe with all required forecasts and variables as input to Microtrader
    
    # Define columns to forecast
    columns_to_forecast = ['open', 'high', 'low', 'close', 'volume', 'transaction_cost']
    
    # Get forecasts for these columns for future steps acc to trades
    forecasted_data = forecast_values(data, trades, columns_to_forecast)
    
    # Add technical indicators this data
    data_w_technical_indicators = add_technical_indicators(data, forecasted_data)
    
    # Add volatility forecasts now
    final_data = add_volatility(data_w_technical_indicators)
    
    return final_data

# -----------------------------------------------------------------------------------------------------

## Environment for Microtrader Layer

In [ ]:
# Trading environment for Microtrader Layer
# Input: (timestamp, # of shares, )

import gym
from gym import spaces
import numpy as np
import pandas as pd

class TradingEnvironmentMicro(gym.Env):
    metadata = {'render.modes': ['human']}
    
    # Data: The combined schedule and forecasted features
    def __init__(self, data):
        super(TradingEnvironment, self).__init__()
        self.data = data
        self.results = []
        self.cumulative_reward = 0
        
        # Extract state columns - have to change according to requirement
        self.state_columns = ['open','high','low','close', 'volume', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 'Stoch_k', 'Stoch_d',
                              'OBV', 'Upper_BB', 'Middle_BB', 'Lower_BB', 'ATR_1', 'ADX', '+DI', '-DI', 'CCI', 'Shares',
                              'transaction_cost'] # Add whatever forecasts you want
        
        
        self.action_space = spaces.Tuple((
            spaces.Discrete(2),  # 0 or 1 for market or limit order
            spaces.Box(low=np.array([0]), high=np.array([1000]), dtype=np.float32)  # Adjust range as needed
        ))
        
        # Observation space definition (example, adjust as needed)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(len(self.state_columns),), dtype=np.float32)

    
    def _get_state(self):
        market_conditions = self._next_observation()
        state = np.append(market_conditions)
        return state
    
    def _next_observation(self):
        return self.data[self.state_columns].iloc[self.current_step].values
    
    def reset(self):
        print('------------------------------------------------Class resetted------------------------------------------------')
        self.current_step = 0
        self.cumulative_reward = 0
        self.results = []
        return self._get_state()
    
    def step(self, action):
        # Adding some noise to the actions
        # action = self._add_noise_to_action(action)
        print(f'Action taken: {action}')

        # type of trade
        trade_type = action[0]        
        price_point = 0 if trade_type == 0 else action[1]
        
        print(f'Debug --> Action: {trade_type} Price: {price_point}')
        
        # Print current and next step for debugging
        print(f'Current step: {self.current_step}')
        self.current_step += 1
        
        # Ensure the index is sequential
        if self.current_step >= len(self.data):
            self.current_step = len(self.data) - 1

        reward = self._calculate_reward(trade_type, price_point)
        self.cumulative_reward += reward
        
        done = self.current_step >= len(self.data)
        if done:
            print(f'Cumulative Rewards: {self.cumulative_reward}')
        
        
        trade_info = {
            'step': self.current_step,
            'timestamp': self.data['timestamp'],
            'action': action,
            'reward': reward
        }
        self.trades.append(trade_info)

        info = {
            'step': self.current_step,
            'action': action,
        }
        

        return self._get_state(), reward, done, info

    
    def _add_noise_to_action(self, action):
        noise = np.random.normal(0, 0.05, size=action.shape)
        action = action + noise
        action = np.clip(action, self.action_space.low, self.action_space.high)
        return action
       
    def _calculate_reward(self, trade_type, price_point):
        # Constants
        pass

    def render(self, mode='human', close=False):
        print('--------------------------------------------------')
        print(f'Steps: {self.current_step}')
        print(f'Cumulative reward: {self.cumulative_reward}')
        self.print_trades()

    def print_trades(self):
        trades_df = pd.DataFrame(self.results)
        for trade in self.results:
            print(f"Step: {trade['step']}, Timestamp: {trade['timestamp']}, Action: {trade['action']}, Reward: {trade['reward']}")
            


## Training & Testing Loop

In [ ]:
# Training Loop for Microtrader

import torch
from stable_baselines3 import PPO
from torch.optim.lr_scheduler import StepLR


# Placeholder data for now
ticker = 'MSFT'  # Specify the ticker you want to trade
ticker_dataa = data[data['symbol'] == ticker]


# Define the learning rate scheduler callback
lr_scheduler_callback = CustomLRScheduler(learning_rate=0.0015, step_size=1000, gamma=0.1)


# Define the best hyperparameters
best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                        'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}

# Create the trading environment
env = TradingEnvironmentMicro(ticker_data,scenario=scenario, preferred_timeframe=timeframe, initial_inventory=transaction_size)

# Initialize the environment and model
model = PPO(CustomTransformerPolicy, env, verbose=1, **best_hyperparameters)

# Train the model
model.learn(total_timesteps=10000, callback=lr_scheduler_callback)

# Save the model
model.save("trading_agent")

# Evaluate the model
obs = env.reset()
for _ in range(len(ticker_data)):
    action, _states = model.predict(obs)
    obs, rewards, done, info = env.step(action)
    if done:
        break

# Render the final state
env.render()